# TP3 - Análisis de Sentimiento sobre Tweets (Sentiment140)
## Notebook 4 — Predicciones con el Modelo Elegido
### Diplomatura IA - UP
### Alumno: Gonzalez Marta Elizabeth
### Mes: Julio26

Este notebook toma el modelo ya **entrenado, validado y evaluado** en los notebooks anteriores (`../data/processed/modelo_final_elegido.pkl`) y lo usa para:

1. Generar predicciones detalladas sobre el conjunto de test (con el texto original, la etiqueta real y la predicha).
2. Mostrar ejemplos concretos de aciertos y errores.
3. Predecir sobre **texto nuevo, arbitrario** — la función que se usaría en un caso de uso real (por ejemplo, un sistema que clasifica tweets nuevos a medida que llegan).

In [1]:
import pandas as pd
import numpy as np
import pickle
from scipy.sparse import hstack, csr_matrix

with open('../data/processed/modelo_final_elegido.pkl', 'rb') as f:
    elegido = pickle.load(f)
nombre_modelo, modelo_final = elegido['nombre'], elegido['modelo']

with open('../data/processed/tfidf_vectorizer_mega.pkl', 'rb') as f:
    tfidf = pickle.load(f)
with open('../data/processed/scaler_features.pkl', 'rb') as f:
    scaler = pickle.load(f)

test = pd.read_csv('../data/processed/split_test.csv')

FEATURE_COLS = [
    'n_chars', 'n_words', 'avg_word_len', 'n_exclamation', 'n_question',
    'n_uppercase_words', 'n_elongated', 'n_pos_emoticons', 'n_neg_emoticons',
    'n_mentions_count', 'n_hashtags_count', 'n_lex_pos', 'n_lex_neg', 'lex_pos_neg_diff',
]

print(f'Modelo cargado: {nombre_modelo}')
print(f'Test: {test.shape}')


Modelo cargado: Regresión Logística + FE
Test: (238910, 30)


## 1. Predicciones detalladas sobre el conjunto de test

Reconstruimos la matriz de features correspondiente al tipo de modelo elegido (algunos usan solo TF-IDF, otros TF-IDF + Feature Engineering).

In [2]:
Xtest_tfidf = tfidf.transform(test['clean_text'])

if nombre_modelo == 'Regresión Logística + FE':
    Xtest_feat = scaler.transform(test[FEATURE_COLS])
    Xtest_final = hstack([Xtest_tfidf, csr_matrix(Xtest_feat)])
else:
    Xtest_final = Xtest_tfidf

pred = modelo_final.predict(Xtest_final)

resultados = test[['text', 'target']].copy()
resultados['prediccion'] = pred
resultados['correcto'] = resultados['target'] == resultados['prediccion']

mapa_etiquetas = {0: 'Negativo', 2: 'Neutral', 4: 'Positivo'}
resultados['target_label'] = resultados['target'].map(mapa_etiquetas)
resultados['prediccion_label'] = resultados['prediccion'].map(mapa_etiquetas)

print(f'Accuracy sobre test: {resultados["correcto"].mean():.2%}')
resultados.head(10)


Accuracy sobre test: 80.75%


,text,target,prediccion,correcto,target_label,prediccion_label
0,@sc430girl i'd let you eat off my plate even,4,4,True,Positivo,Positivo
1,@judel got it! @twithug @judel (hug),4,4,True,Positivo,Positivo
2,@DitaVonTeese I can't believe that,0,0,True,Negativo,Negativo
3,i hate hangovers........,0,0,True,Negativo,Negativo
4,"enjoying healthy dinner, watching Da Vinci Cod...",4,4,True,Positivo,Positivo
5,@iimastarbiitch I would never leave you guys ...,0,0,True,Negativo,Negativo
6,Uhm. Why exactly do we Australians have to wai...,0,0,True,Negativo,Negativo
7,I want green mangoes.,0,0,True,Negativo,Negativo
8,Can't wait to watch jon&amp;kate+8 tonight! I ...,0,0,True,Negativo,Negativo
9,@BirdyLay haa desperate for friends? i wanna v...,4,0,False,Positivo,Negativo


## 2. Ejemplos de aciertos

In [3]:
aciertos = resultados[resultados['correcto']].sample(min(5, resultados['correcto'].sum()), random_state=42)
for _, row in aciertos.iterrows():
    print(f"[{row['target_label']} -> {row['prediccion_label']}] {row['text'][:100]}")


[Positivo -> Positivo] Just got my email - won Euromillions.  Is  12.70 enough to set up your own record label? At least I 
[Positivo -> Positivo] Trying to find a photographer for my sister 
[Negativo -> Negativo] Home. Starting 2 study 4 big exam 2morrow. I'm sad 2day.. nervous bout the exam &amp; from my boy 4 
[Negativo -> Negativo] working...missing my judge shows 
[Positivo -> Positivo] I gotta word tonight at 8:30p.m... so I can sleep the whole day!  until like 6, lol


## 3. Ejemplos de errores

In [4]:
errores = resultados[~resultados['correcto']].sample(min(5, (~resultados['correcto']).sum()), random_state=42)
for _, row in errores.iterrows():
    print(f"[real={row['target_label']} | predicho={row['prediccion_label']}] {row['text'][:100]}")


[real=Positivo | predicho=Negativo] is going to watch Jon and kate plus eight tonight excited to see what is going to happen..... i hope
[real=Negativo | predicho=Positivo] Seriously cannot wait for the end of the week lol.. &amp; Its only just starteeed  lool
[real=Positivo | predicho=Negativo] car garages just know how to rip u off 
[real=Negativo | predicho=Positivo] Photo: first blood drawn! i got bloody making bloody marys.  http://tumblr.com/xhg1z6ysj
[real=Positivo | predicho=Negativo] if u missed me live on kyte earlier check it out at www.kyte.tv/roxxijane. thanks to everyone who st


## 4. Función de predicción para texto nuevo

Esta es la función que se usaría en producción: recibe un tweet nuevo (que el modelo nunca vio), lo limpia con la misma metodología del Notebook 1, y devuelve la predicción.

In [5]:
import re
import html
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english')) - {'not', 'no', 'nor'}
lemmatizer = WordNetLemmatizer()

URL_RE = re.compile(r'https?://\S+|www\.\S+')
MENTION_RE = re.compile(r'@\w+')
HASHTAG_RE = re.compile(r'#(\w+)')
NONALPHA_RE = re.compile(r"[^a-zA-Z\s']")
MULTISPACE_RE = re.compile(r'\s+')

def limpiar_texto(text):
    text = str(text)
    text = html.unescape(text)
    text = URL_RE.sub(' ', text)
    text = MENTION_RE.sub(' ', text)
    text = HASHTAG_RE.sub(r'\1', text)
    text = text.lower()
    text = NONALPHA_RE.sub(' ', text)
    text = MULTISPACE_RE.sub(' ', text).strip()
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 1]
    return ' '.join(tokens)

def predecir_sentimiento(texto_nuevo):
    texto_limpio = limpiar_texto(texto_nuevo)
    X_nuevo = tfidf.transform([texto_limpio])

    if nombre_modelo == 'Regresión Logística + FE':
        # Para un caso de uso real, estas features se calcularían igual que en el Notebook 1
        n_chars = len(texto_nuevo)
        n_words = len(texto_nuevo.split())
        feats = np.array([[n_chars, n_words, 0, texto_nuevo.count('!'), texto_nuevo.count('?'),
                            0, 0, 0, 0, 0, 0, 0, 0, 0]])
        feats_scaled = scaler.transform(feats)
        X_nuevo = hstack([X_nuevo, csr_matrix(feats_scaled)])

    pred = modelo_final.predict(X_nuevo)[0]
    return mapa_etiquetas[pred]

# Ejemplos de prueba con texto inventado
ejemplos_nuevos = [
    "I love this so much, best day ever!!!",
    "This is the worst experience I've ever had, I hate it",
    "The meeting is scheduled for 3pm tomorrow",
]

for texto in ejemplos_nuevos:
    print(f'"{texto}" -> {predecir_sentimiento(texto)}')


"I love this so much, best day ever!!!" -> Positivo
"This is the worst experience I've ever had, I hate it" -> Negativo
"The meeting is scheduled for 3pm tomorrow" -> Negativo


C:\Users\Usuario\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Usuario\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Usuario\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


**Nota:** para el modelo "Regresión Logística + FE", varias features (mayúsculas, elongaciones, emoticones, lexicón) se dejaron en 0 en este ejemplo simplificado — en un uso real conviene reutilizar exactamente las mismas funciones del Notebook 1 (`contar_mayusculas`, `contar_lexicon`, etc.) para calcularlas correctamente sobre el texto nuevo.

## 5. Guardado de las predicciones

In [6]:
resultados.to_csv('../data/processed/predicciones_test.csv', index=False)
print('Guardado: ../data/processed/predicciones_test.csv ->', resultados.shape)


Guardado: ../data/processed/predicciones_test.csv -> (238910, 6)


## 6. Resumen — Notebook 4

1. Se generaron predicciones detalladas sobre el conjunto de test usando el modelo elegido en el Notebook 3, con ejemplos concretos de aciertos y errores.
2. Se armó una función de predicción reutilizable (`predecir_sentimiento`) para clasificar texto nuevo, aplicando la misma limpieza del Notebook 1.
3. Las predicciones completas se exportaron a `../data/processed/predicciones_test.csv`.

**Continúa en `05_topicos_embeddings_grafos.ipynb`** (análisis complementario: tópicos, embeddings, grafos de usuarios).